A Spark UDF (User-Defined Function), ou Função Definida pelo Usuário, é um recurso do Apache Spark que permite criar funções personalizadas em linguagens como Python, Scala, Java ou R para manipular dados em DataFrames ou consultas Spark SQL.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("ExemploUDF").getOrCreate()

# 1. Função Python comum
def formatar_nome(nome):
    if nome:
        return f"Cliente: {nome.strip().title()}"
    return "Cliente Desconhecido"

# 2. Registrar a função como uma Spark UDF
# É necessário definir o tipo de dado de retorno (StringType())
formatar_nome_udf = udf(formatar_nome, StringType())

# 3. Aplicar no DataFrame
df = spark.createDataFrame([("  joao silva ",), ("maria SOUZA",)], ["nome"])
df_resultado = df.withColumn("nome_formatado", formatar_nome_udf(df["nome"]))

df_resultado.show(truncate=False)

+-------------+--------------------+
|nome         |nome_formatado      |
+-------------+--------------------+
|  joao silva |Cliente: Joao Silva |
|maria SOUZA  |Cliente: Maria Souza|
+-------------+--------------------+



In [0]:
df.show()

+-------------+
|         nome|
+-------------+
|  joao silva |
|  maria SOUZA|
+-------------+



Principais Características e Cuidados
Flexibilidade: Permite reaproveitar código e lógica de negócios complexos escritos em Python puro dentro da pipeline de dados.

Gargalo de Performance (Especialmente no PySpark): As UDFs em Python costumam ser lentas. O Spark precisa serializar os dados da JVM (Java Virtual Machine), enviá-los para um processo Python externo para executar a função e depois devolver os dados para a JVM.

Sem Otimização Automática: O otimizador de consultas do Spark (Catalyst Optimizer) trata a UDF como uma "caixa preta", o que impede que ele faça otimizações no plano de execução.

Boas Práticas

Priorize funções nativas: Antes de criar uma UDF, verifique se o módulo pyspark.sql.functions já possui uma solução integrada (como when, concat, upper, etc.). As funções nativas rodam diretamente na JVM e são significativamente mais rápidas.

Pandas UDF (Vectorized UDFs): Se realmente precisar de uma função personalizada em PySpark, prefira usar Pandas UDFs (pyspark.sql.functions.pandas_udf). Elas usam o Apache Arrow para transferir dados em lote, oferecendo uma performance muito superior às UDFs tradicionais.

# NO SQL pelo UDF

UDF diretamente em consultas SQL (spark.sql()), isso precisa registrá-la no catálogo do Spark usando o método spark.udf.register().

Isso torna a função visível e chamável nas suas declarações SQL como se fosse uma função nativa da linguagem.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType

# 1. Iniciar a Spark Session
spark = SparkSession.builder.appName("UDF_SQL_Example").getOrCreate()

# 2. Definir a função em Python puro
def calcular_desconto(preco, percentual):
    if preco is None or percentual is None:
        return 0.0
    return round(preco * (1 - percentual / 100), 2)

# 3. Registrar a UDF no catálogo do Spark SQL
# Parâmetros: ("nome_no_sql", funcao_python, tipo_de_retorno)
spark.udf.register("fn_aplicar_desconto", calcular_desconto, DoubleType())




<function __main__.calcular_desconto(preco, percentual)>

In [0]:
# 4. Criar um DataFrame de exemplo e uma View Temporária
dados = [("Notebook", 3500.0, 10), ("Mouse", 150.0, 5), ("Teclado", 200.0, 15)]
df = spark.createDataFrame(dados, ["produto", "preco", "desconto_pct"])

In [0]:
df.show()

+--------+------+------------+
| produto| preco|desconto_pct|
+--------+------+------------+
|Notebook|3500.0|          10|
|   Mouse| 150.0|           5|
| Teclado| 200.0|          15|
+--------+------+------------+



In [0]:

# A view é necessária para podermos rodar queries SQL na tabela
df.createOrReplaceTempView("produtos")

In [0]:
df.show()

+--------+------+------------+
| produto| preco|desconto_pct|
+--------+------+------------+
|Notebook|3500.0|          10|
|   Mouse| 150.0|           5|
| Teclado| 200.0|          15|
+--------+------+------------+



In [0]:


# 5. Usar a UDF registrada diretamente no spark.sql()
df_resultado = spark.sql("""
    SELECT 
        produto,
        preco,
        desconto_pct,
        fn_aplicar_desconto(preco, desconto_pct) AS preco_com_desconto
    FROM produtos
""")

df_resultado.show()

+--------+------+------------+------------------+
| produto| preco|desconto_pct|preco_com_desconto|
+--------+------+------------+------------------+
|Notebook|3500.0|          10|            3150.0|
|   Mouse| 150.0|           5|             142.5|
| Teclado| 200.0|          15|             170.0|
+--------+------+------------+------------------+



Dica Avançada: Registro via Decorator @udf

Você também pode usar decorators diretamente no Python, o que ajuda a manter o código mais limpo:

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Registrar usando o decorator @udf
@udf(returnType=IntegerType())
def tamanho_texto(texto: str) -> int:
    return len(texto) if texto else 0

# Registrar a UDF no catálogo do Spark SQL
spark.udf.register("produtos_2", tamanho_texto)

# Já pode usar direto no SQL:
spark.sql("SELECT produtos_2('Spark SQL') AS tamanho").show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:102: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


+-------+
|tamanho|
+-------+
|      9|
+-------+



deferença de saprk udf e pandas udf



1. Spark UDF Tradicional

Processa valor por valor.

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# A função recebe e retorna um valor escalar (float)
@udf(returnType=DoubleType())
def dobrar_valor(valor):
    if valor is not None:
        return valor * 2.0
    return None

df = df.withColumn("resultado", dobrar_valor("preco"))

In [0]:
df.show()

+--------+------+------------+---------+
| produto| preco|desconto_pct|resultado|
+--------+------+------------+---------+
|Notebook|3500.0|          10|   7000.0|
|   Mouse| 150.0|           5|    300.0|
| Teclado| 200.0|          15|    400.0|
+--------+------+------------+---------+



2. Pandas UDF (Vetorizada)

Processa uma série inteira de dados do Pandas de uma só vez.

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

# A função recebe e retorna uma Series do Pandas
@pandas_udf(DoubleType())
def dobrar_valor_vetorizado(s: pd.Series) -> pd.Series:
    return s * 2.0  # Operação vetorizada nativa do Pandas/NumPy

df = df.withColumn("resultado2", dobrar_valor_vetorizado("preco"))

In [0]:
df.show()

+--------+------+------------+---------+----------+
| produto| preco|desconto_pct|resultado|resultado2|
+--------+------+------------+---------+----------+
|Notebook|3500.0|          10|   7000.0|    7000.0|
|   Mouse| 150.0|           5|    300.0|     300.0|
| Teclado| 200.0|          15|    400.0|     400.0|
+--------+------+------------+---------+----------+

